# LensIQ: deploy multi-model Roboflow detector to Model Serving

Logs a single MLflow PyFunc that dispatches by `model_id` and proxies
inference to Roboflow's hosted serverless API. The PyFunc is registered
in Unity Catalog and served behind one Databricks Model Serving endpoint
(`lensiq-roboflow-detector` by default), with `ROBOFLOW_API_KEY` injected
as an `environment_var` sourced from a Databricks secret.

Why not bake weights into the artifact: Roboflow's `inference` package
only accepts `project/version` ids for workspaces you own, and the
Universe public models we want either don't allow third-party weight
download or have hosted inference disabled. The PyFunc still gives us a
managed Databricks endpoint (auth, scale-to-zero, logging, billing) while
letting the AppKit server call one consistent API regardless of model.

Payload shape (matches AppKit `serving()` invoke):

```json
{"dataframe_records": [{"image": "<b64>", "model_id": "project/version", "conf": 0.35}]}
```

`model_id` is the `project/version` slug (no workspace) - that's the only
form Roboflow's serverless API routes by. Models must have hosted
inference enabled by their owner; the allowlist baked into `model_config`
controls which ones this endpoint will proxy.

Response shape (matches the YOLO endpoint so `_normalizeDatabricks` works):

```json
{"predictions": [[{"label": "...", "class_id": 0, "confidence": 0.8, "bbox": [x1,y1,x2,y2]}]]}
```

In [ ]:
dbutils.widgets.text("catalog", "iot_dev")
dbutils.widgets.text("schema", "lensiq")
dbutils.widgets.text("registered_name", "lensiq_roboflow_detector")
dbutils.widgets.text("endpoint_name", "lensiq-roboflow-detector")
dbutils.widgets.text("api_key_scope", "reggie_pierce")
dbutils.widgets.text("api_key_secret", "ROBOFLOW_API_KEY")
# Comma-separated allowlist of Roboflow model ids (project/version, no
# workspace prefix - that's what serverless.roboflow.com expects).
dbutils.widgets.text(
    "model_ids",
    ",".join([
        "license-plates-f8vsn/5",
        "spills-ax5xv/2",
        "wet-floor-sign2/1",
        "hard-hat-workers/12",
        "people-detection-o4rdr/7",
    ]),
)

In [ ]:
%pip install -q mlflow>=2.13 pillow numpy requests
dbutils.library.restartPython()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_roboflow")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
REGISTERED = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('registered_name')}"
ENDPOINT = dbutils.widgets.get("endpoint_name")
API_KEY_SCOPE = dbutils.widgets.get("api_key_scope")
API_KEY_SECRET = dbutils.widgets.get("api_key_secret")
MODEL_IDS = [m.strip() for m in dbutils.widgets.get("model_ids").split(",") if m.strip()]
API_KEY = dbutils.secrets.get(scope=API_KEY_SCOPE, key=API_KEY_SECRET)

LOG.info("Deploying multi-model proxy for %d models -> %s", len(MODEL_IDS), ENDPOINT)
for m in MODEL_IDS:
    LOG.info("  - %s", m)

## PyFunc wrapper

Reads `ROBOFLOW_API_KEY` from the serving endpoint's environment (we wire
this in the deploy step below using `EndpointCoreConfigInput.environment_vars`),
then dispatches each input row to the corresponding Roboflow serverless
endpoint. Allowed model ids are baked into `model_config` so a caller
can't trick the endpoint into proxying an arbitrary workspace.

In [ ]:
import json

import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models import infer_signature


class RoboflowMultiDetector(mlflow.pyfunc.PythonModel):
    """PyFunc that proxies inference to Roboflow's hosted serverless API.

    Inputs (per row):
      - image:    base64-encoded JPEG/PNG (with or without `data:` prefix)
      - model_id: Roboflow `project/version` slug (no workspace prefix).
                  Must be in the allowlist passed via
                  `model_config["model_ids"]`.
      - conf:     optional confidence threshold, default 0.35

    Output (per row): list of `{label, class_id, confidence, bbox}` with
    `bbox` as `[x1, y1, x2, y2]` so it matches the YOLO endpoint's contract.
    """

    ROBOFLOW_BASE = "https://serverless.roboflow.com"

    def load_context(self, context):
        import os as _os
        self._api_key = _os.environ.get("ROBOFLOW_API_KEY", "")
        self._allowed = set(json.loads(context.model_config["model_ids"]))

    def _strip_data_url(self, image_b64):
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            return image_b64.split(",", 1)[1]
        return image_b64 or ""

    def _run_one(self, image_b64, model_id, conf):
        if not image_b64 or not model_id:
            return []
        if model_id not in self._allowed:
            return [{"label": "error", "class_id": -1, "confidence": 0.0,
                     "bbox": [0, 0, 0, 0],
                     "error": f"model_id not in allowlist: {model_id}"}]
        if not self._api_key:
            return [{"label": "error", "class_id": -1, "confidence": 0.0,
                     "bbox": [0, 0, 0, 0],
                     "error": "ROBOFLOW_API_KEY env var is not set on endpoint"}]

        import requests
        # Roboflow's REST API expects confidence as a fraction in [0, 1]
        # (matches the OpenAPI default of 0.4). Do NOT scale to percentage -
        # that gets interpreted as a threshold of 35.0 which filters out
        # every prediction.
        c_frac = max(0.01, min(0.99, float(conf if conf is not None else 0.35)))
        url = f"{self.ROBOFLOW_BASE}/{model_id}"
        try:
            resp = requests.post(
                url,
                params={
                    "api_key": self._api_key,
                    "confidence": c_frac,
                    "format": "json",
                },
                headers={"Content-Type": "application/x-www-form-urlencoded"},
                data=self._strip_data_url(image_b64),
                timeout=30,
            )
            resp.raise_for_status()
        except Exception as ex:
            return [{"label": "error", "class_id": -1, "confidence": 0.0,
                     "bbox": [0, 0, 0, 0],
                     "error": f"roboflow http error: {ex}"}]
        body = resp.json()
        out = []
        for p in body.get("predictions", []) or []:
            try:
                cx, cy = float(p["x"]), float(p["y"])
                w, h = float(p["width"]), float(p["height"])
            except (KeyError, TypeError, ValueError):
                continue
            out.append({
                "label": p.get("class") or p.get("class_name") or "object",
                "class_id": int(p.get("class_id", -1)),
                "confidence": float(p.get("confidence", 0.0)),
                "bbox": [
                    int(round(cx - w / 2)),
                    int(round(cy - h / 2)),
                    int(round(cx + w / 2)),
                    int(round(cy + h / 2)),
                ],
            })
        return out

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._run_one(r.get("image"), r.get("model_id"), r.get("conf")) for r in rows]

## Log + register

In [ ]:
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([
    {"image": _TINY_PNG_B64, "model_id": MODEL_IDS[0], "conf": 0.35},
])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

with mlflow.start_run(run_name="deploy_roboflow") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=RoboflowMultiDetector(),
        signature=signature,
        input_example=sample_input,
        registered_model_name=REGISTERED,
        model_config={"model_ids": json.dumps(MODEL_IDS)},
        pip_requirements=[
            "mlflow>=2.13",
            "requests>=2.31",
        ],
    )
LOG.info("Logged model URI: %s", info.model_uri)

## Create / update the serving endpoint

Pulls `ROBOFLOW_API_KEY` from the configured Databricks secret using
`environment_vars` so the served PyFunc reads it from `os.environ` at
load time without baking the value into the artifact.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s -> endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
    environment_vars={
        "ROBOFLOW_API_KEY": f"{{{{secrets/{API_KEY_SCOPE}/{API_KEY_SECRET}}}}}",
    },
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists; updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found; creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for %s; watch Serving UI for readiness.", ENDPOINT)